In [9]:
import os
import sqlite3
import pandas as pd

# ==============================================================================
# 1. WRITE ANSWERS + REFLECTION
# ==============================================================================
WRITEUP_CONTENT = """================================================================================
TASK 1: SQL ANSWERS & REFLECTION RECORD
================================================================================

-- 1. How much is each member borrowing? (Includes zero-checkout members)
SELECT m.member_id, m.name, COUNT(c.checkout_id) AS total_checkouts
FROM members m
LEFT JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, m.name
ORDER BY total_checkouts DESC;

-- 2. Which books match a chosen author pattern? ('Austen')
SELECT book_id, title, author, genre
FROM books
WHERE author LIKE '%Austen%'
ORDER BY title;

-- 3. What are the most popular books? (Top 5)
SELECT b.book_id, b.title, COUNT(c.checkout_id) AS times_borrowed
FROM books b
JOIN checkouts c ON b.book_id = c.book_id
GROUP BY b.book_id, b.title
ORDER BY times_borrowed DESC
LIMIT 5;

-- 4. Who are the most active readers? (Top 10)
SELECT m.member_id, m.name, COUNT(c.checkout_id) AS total_checkouts
FROM members m
JOIN checkouts c ON m.member_id = c.member_id
GROUP BY m.member_id, m.name
ORDER BY total_checkouts DESC
LIMIT 10;

-- 5. Neighborhood activity history ('Downtown', newest 10)
SELECT c.checkout_id, c.checkout_date, m.name, m.neighborhood
FROM checkouts c
JOIN members m ON c.member_id = m.member_id
WHERE m.neighborhood = 'Downtown'
ORDER BY c.checkout_date DESC
LIMIT 10;

================================================================================
REFLECTION: WEB PAGES VS. DATABASES / APIS
================================================================================
1. Structure & Types:
Databases and APIs return structured, strongly-typed data (integers, ISO dates) strictly
governed by schemas. Web pages return unformatted HTML built for human visual layout,
requiring DOM parsing.

2. Integration Friction:
Scraping web page HTML requires extra cleaning (stripping tags, casting strings to IDs/dates,
filling missing fields) so the scraped data matches existing database schema standards
before merging.
"""

with open("answers_and_reflection.txt", "w", encoding="utf-8") as f:
    f.write(WRITEUP_CONTENT)
print("✓ Wrote answers_and_reflection.txt")


# ==============================================================================
# 2. DATA COMBINATION PIPELINE
# ==============================================================================
DB_FILE = "library.db"
HTML_FILE = "reading_kickoff.html"

for path in (DB_FILE, HTML_FILE):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Cannot find '{path}' in {os.getcwd()}")

# --- Load DB tables ---
with sqlite3.connect(DB_FILE) as conn:
    members = pd.read_sql("SELECT * FROM members", conn)
    checkouts = pd.read_sql("SELECT * FROM checkouts", conn)
    books = pd.read_sql("SELECT * FROM books", conn)

members["member_total_checkouts"] = (
    members["member_id"].map(checkouts["member_id"].value_counts()).fillna(0).astype(int)
)

# --- Merge checkouts -> members -> books ---
db_df = checkouts.merge(members, on="member_id", how="left").merge(books, on="book_id", how="left")

# --- Parse the HTML table (pandas does the scraping in one line) ---
kickoff_df = pd.read_html(HTML_FILE)[0]
kickoff_df.columns = [c.strip().lower().replace(" ", "_") for c in kickoff_df.columns]
kickoff_df = kickoff_df.rename(columns={"date": "checkout_date", "member_name": "name", "book_title": "title"})

# --- Combine everything and clean types ---
final_dataset = pd.concat([db_df, kickoff_df], ignore_index=True)

for col in ["member_id", "book_id", "checkout_id", "member_total_checkouts"]:
    if col in final_dataset.columns:
        final_dataset[col] = pd.to_numeric(final_dataset[col], errors="coerce").astype("Int64")

final_dataset.to_csv("combined_library_dataset.csv", index=False)
print(f"✓ Wrote combined_library_dataset.csv ({len(final_dataset)} rows)")

✓ Wrote answers_and_reflection.txt
✓ Wrote combined_library_dataset.csv (417 rows)


In [10]:
import os
import pandas as pd
import numpy as np

# -----------------------------------------------------------------------------
# 0. LOAD DATA
# -----------------------------------------------------------------------------
combined_file = "combined_library_dataset.csv"
members_file = "registered_members.csv"

if not os.path.exists(combined_file):
    raise FileNotFoundError(f"'{combined_file}' not found in {os.getcwd()}")

df = pd.read_csv(combined_file)
print(f"Loaded '{combined_file}': {df.shape[0]} rows, {df.shape[1]} columns\n")

valid_member_ids = None
if os.path.exists(members_file):
    valid_member_ids = set(pd.read_csv(members_file)["member_id"].dropna().unique())
    print(f"Loaded '{members_file}': {len(valid_member_ids)} valid member IDs indexed.\n")
else:
    print(f"Notice: '{members_file}' not found. Problem 4 check requires this file.\n")

report = {}

# -----------------------------------------------------------------------------
# PROBLEM 1: MISSING VALUES
# -----------------------------------------------------------------------------
print("--- PROBLEM 1: MISSING VALUES ---")
missing = df.isnull().sum()
missing = missing[missing > 0]

report["p1"] = {}
for col, count in missing.items():
    pct = round(count / len(df) * 100, 2)
    report["p1"][col] = {"count": int(count), "pct": pct}
    print(f"Column '{col}': {count} missing records ({pct:.2f}%)")

    if df[col].dtype == "object":
        df[col] = df[col].fillna("Unknown")          # categorical: keep row, mark unknown
    elif np.issubdtype(df[col].dtype, np.number):
        df[col] = df[col].fillna(df[col].median())   # numeric: median is outlier-resistant

if missing.empty:
    print("No missing values found in any column.")

# -----------------------------------------------------------------------------
# PROBLEM 2: DUPLICATE RECORDS
# -----------------------------------------------------------------------------
print("\n--- PROBLEM 2: DUPLICATE RECORDS ---")
dupes = int(df.duplicated().sum())
report["p2_exact_duplicates"] = dupes
print(f"True full-row duplicates detected: {dupes}")

df = df.drop_duplicates(keep="first").reset_index(drop=True)
print(f"Rows remaining after dropping true duplicates: {len(df)}")

# -----------------------------------------------------------------------------
# PROBLEM 3: TEXT INCONSISTENCIES
# -----------------------------------------------------------------------------
print("\n--- PROBLEM 3: TEXT INCONSISTENCIES ---")
standardization_map = {
    "downtown": "Downtown", "down town": "Downtown", "DOWNTOWN": "Downtown",
    "north side": "Northside", "North Side": "Northside", "NORTHSIDE": "Northside",
    "west end": "West End", "West end": "West End",
    "active": "Active", "ACTIVE": "Active", "actv": "Active",
    "inactive": "Inactive", "INACTIVE": "Inactive",
    "cancelled": "Canceled", "canceled": "Canceled", "CANCELED": "Canceled",
}

report["p3"] = {}
for col in df.select_dtypes(include="object").columns:
    before = df[col].nunique()
    df[col] = df[col].astype(str).str.strip().replace(standardization_map)
    after = df[col].nunique()

    if before != after:
        report["p3"][col] = {"before": before, "after": after}
        print(f"Column '{col}': Standardized variants from {before} down to {after} distinct values.")

# -----------------------------------------------------------------------------
# PROBLEM 4: ORPHANED / INVALID MEMBER IDs
# -----------------------------------------------------------------------------
print("\n--- PROBLEM 4: REFERENTIAL INTEGRITY (member_id) ---")
if "member_id" not in df.columns:
    print("Error: 'member_id' column not present in the combined dataset.")
elif valid_member_ids is None:
    print("Skipped: 'registered_members.csv' not found. Re-run after placing the file in directory.")
else:
    invalid_mask = ~df["member_id"].isin(valid_member_ids)
    report["p4_invalid_members"] = int(invalid_mask.sum())
    print(f"Orphaned activity records with non-existent member_ids: {invalid_mask.sum()}")

    df = df[~invalid_mask].reset_index(drop=True)
    print(f"Rows remaining after removing invalid member_ids: {len(df)}")

# -----------------------------------------------------------------------------
# OUTPUT
# -----------------------------------------------------------------------------
output_path = "task2_cleaned_data.csv"
df.to_csv(output_path, index=False)

print("\n" + "=" * 60)
print(f"SUCCESS: Cleaned dataset exported to '{output_path}'")
print(f"Final shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("=" * 60)

Loaded 'combined_library_dataset.csv': 417 rows, 14 columns

Notice: 'registered_members.csv' not found. Problem 4 check requires this file.

--- PROBLEM 1: MISSING VALUES ---
Column 'checkout_id': 26 missing records (6.24%)
Column 'return_date': 91 missing records (21.82%)
Column 'first_name': 26 missing records (6.24%)
Column 'last_name': 26 missing records (6.24%)
Column 'grade': 62 missing records (14.87%)
Column 'neighborhood': 26 missing records (6.24%)
Column 'membership_status': 26 missing records (6.24%)
Column 'join_date': 31 missing records (7.43%)
Column 'member_total_checkouts': 26 missing records (6.24%)
Column 'title': 26 missing records (6.24%)
Column 'author': 26 missing records (6.24%)

--- PROBLEM 2: DUPLICATE RECORDS ---
True full-row duplicates detected: 8
Rows remaining after dropping true duplicates: 409

--- PROBLEM 3: TEXT INCONSISTENCIES ---
Column 'neighborhood': Standardized variants from 10 down to 9 distinct values.
Column 'membership_status': Standardized

In [11]:
!pip install python-docx --quiet

import pandas as pd
from docx import Document

# 1. Load data and calculate neighborhood summary
df = pd.read_csv("task2_cleaned_data.csv")

required_cols = {"neighborhood", "member_id", "checkout_id"}
missing = required_cols - set(df.columns)
if missing:
    raise KeyError(f"Missing expected column(s) in CSV: {missing}. Found columns: {list(df.columns)}")

summary = (
    df.groupby("neighborhood")
    .agg(
        member_count=("member_id", "nunique"),
        total_checkouts=("checkout_id", "count"),
    )
    .reset_index()
)
summary["checkouts_per_member"] = (
    summary["total_checkouts"] / summary["member_count"]
).round(2)
summary["member_share_pct"] = (
    summary["member_count"] / summary["member_count"].sum() * 100
).round(1)
summary["checkout_share_pct"] = (
    summary["total_checkouts"] / summary["total_checkouts"].sum() * 100
).round(1)
lowest = summary.sort_values("checkouts_per_member").iloc[0]

# 2. Build Document
doc = Document()
doc.add_heading("Data Fairness Reflection", level=0)
doc.add_paragraph(
    "Dataset: task2_cleaned_data.csv | Project: Library Reading Program Analysis"
)

sections = [
    (
        "1. Evaluation Standard (Basis)",
        "Representation is measured by comparing each neighborhood's share of total members against "
        "its share of total checkouts and per-capita borrowing rate. A neighborhood is under-represented "
        "if its checkout rate per member falls significantly below average.",
    ),
    (
        "2. Quantitative Finding",
        f"Based on the analysis, {lowest['neighborhood']} shows clear under-representation. "
        f"While representing {lowest['member_share_pct']}% of registered members ({lowest['member_count']} members), "
        f"it accounts for only {lowest['checkout_share_pct']}% of checkouts ({lowest['total_checkouts']} checkouts), "
        f"averaging {lowest['checkouts_per_member']} checkouts per member.",
    ),
]
for title, text in sections:
    doc.add_heading(title, level=2)
    doc.add_paragraph(text)

# Table
table = doc.add_table(rows=1, cols=5)
table.style = "Table Grid"
headers = [
    "Neighborhood",
    "Members",
    "Checkouts",
    "Checkouts/Member",
    "% Checkout Share",
]
for i, h in enumerate(headers):
    table.rows[0].cells[i].text = h
for _, row in summary.iterrows():
    cells = table.add_row().cells
    cells[0].text = str(row["neighborhood"])
    cells[1].text = str(row["member_count"])
    cells[2].text = str(row["total_checkouts"])
    cells[3].text = str(row["checkouts_per_member"])
    cells[4].text = f"{row['checkout_share_pct']}%"

# Remaining Sections
doc.add_heading("3. Plausible Reason for Disparity", level=2)
doc.add_paragraph(
    f"Lower borrowing in {lowest['neighborhood']} may stem from transit barriers, limited operating hours, "
    "and reduced marketing outreach during the event."
)
doc.add_heading("4. Actionable Next Step for Next Summer", level=2)
doc.add_paragraph(
    f"Deploy a targeted Mobile Pop-Up Library in {lowest['neighborhood']} community centers and partner "
    "with local schools to distribute kickoff kits directly."
)

doc.save("fairness_reflection.docx")
print("Generated: fairness_reflection.docx")

Generated: fairness_reflection.docx
